In [1]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 72.2 MB/s eta 0:00:00


In [23]:
import os, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, re
import plotly.graph_objects as go
import streamlit as st
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from streamlit_option_menu import option_menu

# ============================================================
# 🚀 1. SYSTEM ENVIRONMENT & THEME CONFIGURATION
# ============================================================
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#5D3FD3"\nbackgroundColor="#F8F9FA"\nsecondaryBackgroundColor="#E0E7FF"\ntextColor="#212529"\n')

st.set_page_config(page_title="Infosys Portal", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#F8F9FA", "bg_sidebar": "#E0E7FF", "bg_card": "#FFFFFF", "bg_card_alt": "#CED4DA",
    "text_main": "#212529", "text_heading": "#343A40", "text_muted": "#6C757D",
    "accent": "#5D3FD3", "accent_hover": "#482EA8", "accent_text": "#FFFFFF",
    "border": "#343A40", "border_light": "#ADB5BD", "success": "#28A745", "danger": "#DC3545"
}

JWT_SECRET = "super-secret-infosys-key-2026"
SENDER_EMAIL = "tazreenrahman024@gmail.com"
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")  # Pulled securely from Colab Secrets!
OTP_EXPIRY_MINUTES = 5

# ============================================================
# 🎨 2. NEO-BRUTALIST PREMIUM USER INTERFACE STYLING (CSS)
# ============================================================
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');
    html, body, .stApp {{ background: {COLORS['bg_main']} !important; font-family: 'Inter', sans-serif !important; color: {COLORS['text_main']} !important; }}\n
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}\n    header {{ background: transparent !important; z-index: 999999 !important; }}

    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['accent']} !important; border: 2px solid {COLORS['border']} !important;
        border-radius: 8px !important; padding: 6px !important; margin: 8px !important;
        box-shadow: 3px 3px 0px {COLORS['border']} !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['accent_text']} !important; color: {COLORS['accent_text']} !important; stroke: {COLORS['accent_text']} !important;
    }}

    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; }}

    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{ background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border_light']} !important; border-radius: 10px !important; }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; }}
    input, div[data-baseweb="select"] span {{ color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important; }}

    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important; color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important; border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 14px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 16px !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important; transform: translate(-2px, -2px) !important;
        box-shadow: 6px 6px 0px {COLORS['border']} !important;
    }}
    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 2px solid {COLORS['border']} !important; }}
    .pn-card {{ background: {COLORS['bg_card']}; border: 2px solid {COLORS['border_light']}; border-radius: 14px; padding: 24px; box-shadow: 4px 4px 0px {COLORS['border_light']}; }}
</style>
""", unsafe_allow_html=True)

# ============================================================
# 💾 3. DATABASE INFRASTRUCTURE & PRE-SEEDING
# ============================================================
def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    # Pre-seed Admin Bypass Account
    if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                     ("Administrator", "infosys@ai", hash_txt("admin@123"), "What is your pet name?", hash_txt("admin")))

# ============================================================
# 🛡️ 4. STEP 8 FIELD VALIDATION ENGINE (REGEX)
# ============================================================
def validate_email(email_str):
    # Rule: At least 2 chars before @, 2 between @ and dot, 2 after dot (ab@cd.ef)
    pattern = r"^[a-zA-Z0-9._%+-]{2,}@[a-zA-Z0-9.-]{2,}\.[a-zA-Z]{2,}$"
    return re.match(pattern, email_str) is not None

def validate_password(pwd_str):
    # Rule: Min 8 chars, 1 upper, 1 lower, 1 number, 1 special character
    if len(pwd_str) < 8: return False
    if not re.search(r"[A-Z]", pwd_str): return False
    if not re.search(r"[a-z]", pwd_str): return False
    if not re.search(r"[0-9]", pwd_str): return False
    if not re.search(r"[_@#$%^&+=!¡?¿*()<>[\\]{}|\-.:;,+~`'\"\\s]", pwd_str): return False
    return True

# ============================================================
# 📧 5. CRYPTOGRAPHIC OTP & JWT GENERATORS
# ============================================================
def generate_otp(): return f"{secrets.randbelow(900000) + 100000}"

def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

def make_otp_token(email, otp):
    payload = {
        "sub": email, "otp_hash": hash_txt(otp), "type": "password_reset_otp",
        "iat": datetime.datetime.utcnow(), "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email or payload.get("type") != "password_reset_otp":
            return False, "Security token mismatch."
        if check_txt(input_otp, payload["otp_hash"]):
            return True, "Valid"
        return False, "Invalid 6-digit OTP code."
    except jwt.ExpiredSignatureError:
        return False, f"⚠️ This OTP code expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new one."
    except Exception:
        return False, "Invalid or corrupted verification token."

def send_professional_email(to_email, otp, app_pass):
    msg = MIMEMultipart('alternative')
    msg['From'] = f"Infosys Support <{SENDER_EMAIL}>"
    msg['To'] = to_email
    msg['Subject'] = "Infosys Portal - Verification Code"
    msg['Date'] = formatdate(localtime=True)
    msg['Message-ID'] = make_msgid()
    msg['Reply-To'] = SENDER_EMAIL

    text_body = f"Your verification code for Infosys Portal is: {otp}\nThis code will expire in {OTP_EXPIRY_MINUTES} minutes."
    html_body = f"""
    <html><body><div style="font-family:Arial,sans-serif;max-width:500px;margin:0 auto;border:2px solid #272343;border-radius:12px;padding:30px;text-align:center;">
        <h2 style="color:#272343;">Infosys Portal Verification</h2>
        <p>Please use the password recovery verification code below:</p>
        <div style="background-color:#ffd803;color:#272343;font-size:28px;font-weight:bold;letter-spacing:5px;padding:15px;border:2px solid #272343;border-radius:8px;display:inline-block;margin:10px 0;">{otp}</div>
        <p>This code expires in <b>{OTP_EXPIRY_MINUTES} minutes</b>.</p>
    </div></body></html>
    """
    msg.attach(MIMEText(text_body, 'plain'))
    msg.attach(MIMEText(html_body, 'html'))
    try:
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls()
        s.login(SENDER_EMAIL, app_pass if app_pass else "")
        s.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        s.quit()
        return True, "Email sent successfully!"
    except Exception as e:
        return False, f"SMTP Error: {str(e)}"

# ============================================================
# 🔄 6. APPLICATION ROUTING & STATE MANAGEMENT
# ============================================================
for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None), ("forgot_stage", "init"), ("jwt_otp_token", "")]:
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">Infosys Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# 🗺️ 7. VISUAL VIEWPORT NAVIGATION PIPELINE
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        # 🔒 STANDARD SIGN IN PAGE
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            email_inp = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd_inp = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True, key="login_submit_btn"):
                if not email_inp or not pwd_inp:
                    st.error("⚠️ All fields are mandatory.")
                else:
                    with get_db() as c: r = c.execute("SELECT password_hash FROM users WHERE email=?", (email_inp,)).fetchone()
                    if r and check_txt(pwd_inp, r[0]):
                        st.session_state.token = make_jwt(email_inp)
                        navigate("Dashboard")
                    else:
                        st.error("❌ Invalid credentials.")
            if col_c.button("Create Account", use_container_width=True, key="goto_signup_btn"): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True, key="goto_forgot_btn"): navigate("Forgot")

        # 📝 STEP 8 COMPLIANT REGISTRATION PAGE
        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Portal today")
            uname = st.text_input("Full name / Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm new password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True, key="signup_submit_btn"):
                if not uname or not email or not pwd or not confirm_pwd or not sa:
                    st.error("⚠️ All fields are mandatory.")
                elif not validate_email(email):
                    st.error("❌ Invalid Email layout (Needs structure format like ab@cd.ef).")
                elif not validate_password(pwd):
                    st.error("❌ Password weak! Must have 8+ characters, 1 uppercase, 1 lowercase, 1 number, and 1 symbol.")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip())))
                        st.session_state.token = make_jwt(email)
                        st.success("✅ Account created successfully!")
                        time.sleep(1)
                        navigate("Dashboard")
                    except sqlite3.IntegrityError:
                        st.error("❌ Email or Username already registered inside ledger.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True, key="back_to_login_signup"): navigate("Login")

        # 🔐 STEP 9 COMPLIANT DUAL RECOVERY FLOW (SECURITY QUESTION & OTP RELAY)
        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")

            if st.session_state.forgot_stage == "init":
                email_target = st.text_input("Registered email address", placeholder="you@infosys.com").lower().strip()
                st.markdown("<br>", unsafe_allow_html=True)

                col_sq, col_otp = st.columns(2)
                if col_sq.button("Via Security Question", use_container_width=True, key="forgot_route_sq_btn"):
                    if not email_target: st.error("⚠️ Please enter your email address.")
                    else:
                        with get_db() as c: r = c.execute("SELECT security_question FROM users WHERE email=?", (email_target,)).fetchone()
                        if r:
                            st.session_state.reset_email = email_target
                            st.session_state.sq_p = r[0]
                            st.session_state.reset_mode = "sq"
                            st.session_state.forgot_stage = "challenge"
                            st.rerun()
                        else: st.error("❌ Registered email address not found.")

                if col_otp.button("Via OTP Mail Relay", use_container_width=True, key="forgot_route_otp_btn"):
                    if not email_target: st.error("⚠️ Please enter your email address.")
                    elif not EMAIL_PASSWORD:
                        st.error("❌ Gmail App Password missing! Put `EMAIL_PASSWORD` inside Colab Secrets.")
                    else:
                        with get_db() as c: r = c.execute("SELECT 1 FROM users WHERE email=?", (email_target,)).fetchone()
                        if r:
                            otp_code = generate_otp()
                            with st.spinner("Dispatching verification OTP over secure SMTP relays..."):
                                ok, msg = send_professional_email(email_target, otp_code, EMAIL_PASSWORD)
                            if ok:
                                st.session_state.reset_email = email_target
                                st.session_state.jwt_otp_token = make_otp_token(email_target, otp_code)
                                st.session_state.reset_mode = "otp"
                                st.session_state.forgot_stage = "challenge"
                                st.success("✅ Code dispatched! Check your inbox.")
                                time.sleep(1)
                                st.rerun()
                            else: st.error(f"❌ {msg}")
                        else: st.error("❌ Registered email address not found.")

            else:
                # --- PROCESS ROUTE A (SECURITY CHALLENGE) ---
                if st.session_state.reset_mode == "sq":
                    st.info(f"❓ **Security Question Challenge:** {st.session_state.sq_p}")
                    ans = st.text_input("Your answer", placeholder="Enter answer given at registration").lower().strip()
                    npw = st.text_input("New password", type="password", placeholder="Min. 8 characters")
                    confirm_npw = st.text_input("Confirm new password", type="password", placeholder="Re-enter password")
                    st.markdown("<br>", unsafe_allow_html=True)

                    if st.button("Reset Password via Question →", use_container_width=True, key="sq_reset_execute_btn"):
                        if not ans or not npw or not confirm_npw: st.error("⚠️ All fields are mandatory.")
                        elif not validate_password(npw): st.error("❌ Password failure. Must fulfill Step 8 criteria rules.")
                        elif npw != confirm_npw: st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c: r = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                            if r and check_txt(ans, r[0]):
                                with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                st.success("✅ Credentials verified. Password updated successfully!")
                                time.sleep(1.5)
                                st.session_state.reset_email = None
                                st.session_state.forgot_stage = "init"
                                navigate("Login")
                            else: st.error("❌ Incorrect security answer.")

                # --- PROCESS ROUTE B (OTP TOKEN CHALLENGE) ---
                elif st.session_state.reset_mode == "otp":
                    st.info(f"📧 Code sent to **{st.session_state.reset_email}** (Expires in {OTP_EXPIRY_MINUTES} mins).")
                    otp_input = st.text_input("6-Digit OTP Code", placeholder="e.g. 123456").strip()
                    npw = st.text_input("New password", type="password", placeholder="Min. 8 characters")
                    confirm_npw = st.text_input("Confirm new password", type="password", placeholder="Re-enter password")
                    st.markdown("<br>", unsafe_allow_html=True)

                    if st.button("Verify OTP & Reset Password →", use_container_width=True, key="otp_reset_execute_btn"):
                        if not otp_input or not npw or not confirm_npw: st.error("⚠️ All fields are mandatory.")
                        elif not validate_password(npw): st.error("❌ Password failure. Must fulfill Step 8 criteria rules.")
                        elif npw != confirm_npw: st.error("❌ Passwords do not match.")
                        else:
                            ok, msg = verify_otp_token(st.session_state.jwt_otp_token, otp_input, st.session_state.reset_email)
                            if ok:
                                with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                st.success("🎉 OTP verified. Account credentials updated successfully!")
                                time.sleep(1.5)
                                st.session_state.reset_email = None
                                st.session_state.forgot_stage = "init"
                                navigate("Login")
                            else: st.error(f"❌ {msg}")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel Recovery Process", use_container_width=True, key="cancel_recovery_btn"):
                st.session_state.reset_email = None
                st.session_state.reset_mode = None
                st.session_state.forgot_stage = "init"
                navigate("Login")

# ============================================================
# 📊 8. AUTHENTICATED PORTALS (STEP 10 & 11 SEPARATION)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    current_email = payload["email"]
    with get_db() as c: user_row = c.execute("SELECT username FROM users WHERE email=?", (current_email,)).fetchone()
    uname = user_row[0] if user_row else "Core User"

    # 🚀 SIDEBAR NAVIGATION WORKSPACE & LOGOUT GATE
    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⚡</div>
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Infosys Portal</div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Management Panel" if current_email=="infosys@ai" else "Intelligent Analytics"}</div>
        </div><hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Settings", "Logout"] if current_email=="infosys@ai" else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "gear", "box-arrow-right"] if current_email=="infosys@ai" else ["house", "graph-up", "file-text", "box-arrow-right"],
                           styles={"container": {"background-color": COLORS['bg_sidebar']}, "nav-link-selected": {"background-color": COLORS['accent'], "color": COLORS['accent_text']}})

        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.success("🔒 Session cleared safely. Returning to sign in screen...")
            time.sleep(1)
            st.rerun()

    # 🛡️ STEP 11: PRIVILEGED ADMINISTRATIVE MASTER DASHBOARD
    if current_email == "infosys@ai":
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        st.subheader("👥 Registered Platform Accounts Ledger")

        # Read-only fetch: Username/Email only. Never select password_hash!
        with get_db() as c: user_data = c.execute("SELECT id, username, email FROM users").fetchall()

        if user_data:
            st.markdown(f"""
            <div style="background:#ffffff; border:2px solid {COLORS['border_light']}; border-radius:12px; padding:24px;">
                <div style="display:grid; grid-template-columns: 1fr 3fr 4fr; font-weight:700; color:{COLORS['text_heading']}; border-bottom:2px solid #e2e8f0; padding-bottom:10px; margin-bottom:12px;">
                    <div>User ID</div>
                    <div>Username</div>
                    <div>Email Endpoint Address</div>
                </div>
            """, unsafe_allow_html=True)

            for row in user_data:
                col_id, col_un, col_em = st.columns([1, 3, 4])
                col_id.markdown(f"**`#{row[0]}`**")
                col_un.write(row[1])
                col_em.write(row[2])
                st.markdown("<hr style='margin:8px 0; border-color:#f1f5f9;'>", unsafe_allow_html=True)

            st.markdown("</div>", unsafe_allow_html=True)
            st.markdown("<br>", unsafe_allow_html=True)
            st.caption(f"📊 Total System Accounts Registered: **{len(user_data)}**")
        else:
            st.info("ℹ️ No users registered inside the database yet.")

    # 📈 STEP 10: METRIC ANALYTICS STANDARD PORTAL VIEWPORT
    else:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Analytics Dashboard</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:28px;">{icon}</div>
                <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['accent']}, "bgcolor": COLORS['bg_card_alt'], "borderwidth": 1, "bordercolor": COLORS['border']}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)

2026-07-07 09:10:13.721 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 09:10:13.724 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 09:10:13.725 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 09:10:13.726 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 09:10:13.737 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 09:10:13.738 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 09:10:13.740 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-07 09:10:13.740 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [27]:
# Create app.py from the content of the Streamlit application cell
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, re
import plotly.graph_objects as go
import streamlit as st
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from streamlit_option_menu import option_menu

# ============================================================
# 🚀 1. SYSTEM ENVIRONMENT & THEME CONFIGURATION
# ============================================================
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#5D3FD3"\nbackgroundColor="#F8F9FA"\nsecondaryBackgroundColor="#E0E7FF"\ntextColor="#212529"\n')

st.set_page_config(page_title="Infosys Portal", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#F8F9FA", "bg_sidebar": "#E0E7FF", "bg_card": "#FFFFFF", "bg_card_alt": "#CED4DA",
    "text_main": "#212529", "text_heading": "#343A40", "text_muted": "#6C757D",
    "accent": "#5D3FD3", "accent_hover": "#482EA8", "accent_text": "#FFFFFF",
    "border": "#343A40", "border_light": "#ADB5BD", "success": "#28A745", "danger": "#DC3545"
}

JWT_SECRET = "super-secret-infosys-key-2026"
SENDER_EMAIL = "tazreenrahman024@gmail.com"
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")  # Pulled securely from Colab Secrets!
OTP_EXPIRY_MINUTES = 5

# ============================================================
# 🎨 2. NEO-BRUTALIST PREMIUM USER INTERFACE STYLING (CSS)
# ============================================================
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');
    html, body, .stApp {{ background: {COLORS['bg_main']} !important; font-family: 'Inter', sans-serif !important; color: {COLORS['text_main']} !important; }}\n
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}\n    header {{ background: transparent !important; z-index: 999999 !important; }}

    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['accent']} !important; border: 2px solid {COLORS['border']} !important;
        border-radius: 8px !important; padding: 6px !important; margin: 8px !important;
        box-shadow: 3px 3px 0px {COLORS['border']} !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['accent_text']} !important; color: {COLORS['accent_text']} !important; stroke: {COLORS['accent_text']} !important;
    }}

    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; }}

    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{ background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border_light']} !important; border-radius: 10px !important; }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; }}
    input, div[data-baseweb="select"] span {{ color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important; }}

    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important; color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important; border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 14px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 16px !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important; transform: translate(-2px, -2px) !important;
        box-shadow: 6px 6px 0px {COLORS['border']} !important;
    }}
    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 2px solid {COLORS['border']} !important; }}
    .pn-card {{ background: {COLORS['bg_card']}; border: 2px solid {COLORS['border_light']}; border-radius: 14px; padding: 24px; box-shadow: 4px 4px 0px {COLORS['border_light']}; }}
</style>
""", unsafe_allow_html=True)

# ============================================================
# 💾 3. DATABASE INFRASTRUCTURE & PRE-SEEDING
# ============================================================
def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    # Pre-seed Admin Bypass Account
    if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                     ("Administrator", "infosys@ai", hash_txt("admin@123"), "What is your pet name?", hash_txt("admin")))

# ============================================================
# 🛡️ 4. STEP 8 FIELD VALIDATION ENGINE (REGEX)
# ============================================================
def validate_email(email_str):
    # Rule: At least 2 chars before @, 2 between @ and dot, 2 after dot (ab@cd.ef)
    pattern = r"^[a-zA-Z0-9._%+-]{2,}@[a-zA-Z0-9.-]{2,}\.[a-zA-Z]{2,}$"
    return re.match(pattern, email_str) is not None

def validate_password(pwd_str):
    # Rule: Min 8 chars, 1 upper, 1 lower, 1 number, 1 special character
    if len(pwd_str) < 8: return False
    if not re.search(r"[A-Z]", pwd_str): return False
    if not re.search(r"[a-z]", pwd_str): return False
    if not re.search(r"[0-9]", pwd_str): return False
    if not re.search(r"[_@#$%^&+=!¡?¿*()<>[\\]{}|\-.:;,+~`'\"\\s]", pwd_str): return False
    return True

# ============================================================
# 📧 5. CRYPTOGRAPHIC OTP & JWT GENERATORS
# ============================================================
def generate_otp(): return f"{secrets.randbelow(900000) + 100000}"

def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

def make_otp_token(email, otp):
    payload = {
        "sub": email, "otp_hash": hash_txt(otp), "type": "password_reset_otp",
        "iat": datetime.datetime.utcnow(), "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email or payload.get("type") != "password_reset_otp":
            return False, "Security token mismatch."
        if check_txt(input_otp, payload["otp_hash"]):
            return True, "Valid"
        return False, "Invalid 6-digit OTP code."
    except jwt.ExpiredSignatureError:
        return False, f"⚠️ This OTP code expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new one."
    except Exception:
        return False, "Invalid or corrupted verification token."

def send_professional_email(to_email, otp, app_pass):
    msg = MIMEMultipart('alternative')
    msg['From'] = f"Infosys Support <{SENDER_EMAIL}>"
    msg['To'] = to_email
    msg['Subject'] = "Infosys Portal - Verification Code"
    msg['Date'] = formatdate(localtime=True)
    msg['Message-ID'] = make_msgid()
    msg['Reply-To'] = SENDER_EMAIL

    text_body = f"Your verification code for Infosys Portal is: {otp}\nThis code will expire in {OTP_EXPIRY_MINUTES} minutes."
    html_body = f"""
    <html><body><div style="font-family:Arial,sans-serif;max-width:500px;margin:0 auto;border:2px solid #272343;border-radius:12px;padding:30px;text-align:center;">
        <h2 style="color:#272343;">Infosys Portal Verification</h2>
        <p>Please use the password recovery verification code below:</p>
        <div style="background-color:#ffd803;color:#272343;font-size:28px;font-weight:bold;letter-spacing:5px;padding:15px;border:2px solid #272343;border-radius:8px;display:inline-block;margin:10px 0;">{otp}</div>
        <p>This code expires in <b>{OTP_EXPIRY_MINUTES} minutes</b>.</p>
    </div></body></html>
    """
    msg.attach(MIMEText(text_body, 'plain'))
    msg.attach(MIMEText(html_body, 'html'))
    try:
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls()
        s.login(SENDER_EMAIL, app_pass if app_pass else "")
        s.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        s.quit()
        return True, "Email sent successfully!"
    except Exception as e:
        return False, f"SMTP Error: {str(e)}"

# ============================================================
# 🔄 6. APPLICATION ROUTING & STATE MANAGEMENT
# ============================================================
for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None), ("forgot_stage", "init"), ("jwt_otp_token", "")]:
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">Infosys Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# 🗺️ 7. VISUAL VIEWPORT NAVIGATION PIPELINE
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        # 🔒 STANDARD SIGN IN PAGE
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            email_inp = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd_inp = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True, key="login_submit_btn"):
                if not email_inp or not pwd_inp:
                    st.error("⚠️ All fields are mandatory.")
                else:
                    with get_db() as c: r = c.execute("SELECT password_hash FROM users WHERE email=?", (email_inp,)).fetchone()
                    if r and check_txt(pwd_inp, r[0]):
                        st.session_state.token = make_jwt(email_inp)
                        navigate("Dashboard")
                    else:
                        st.error("❌ Invalid credentials.")
            if col_c.button("Create Account", use_container_width=True, key="goto_signup_btn"): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True, key="goto_forgot_btn"): navigate("Forgot")

        # 📝 STEP 8 COMPLIANT REGISTRATION PAGE
        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Portal today")
            uname = st.text_input("Full name / Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm new password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True, key="signup_submit_btn"):
                if not uname or not email or not pwd or not confirm_pwd or not sa:
                    st.error("⚠️ All fields are mandatory.")
                elif not validate_email(email):
                    st.error("❌ Invalid Email layout (Needs structure format like ab@cd.ef).")
                elif not validate_password(pwd):
                    st.error("❌ Password weak! Must have 8+ characters, 1 uppercase, 1 lowercase, 1 number, and 1 symbol.")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip())))
                        st.session_state.token = make_jwt(email)
                        st.success("✅ Account created successfully!")
                        time.sleep(1)
                        navigate("Dashboard")
                    except sqlite3.IntegrityError:
                        st.error("❌ Email or Username already registered inside ledger.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True, key="back_to_login_signup"): navigate("Login")

        # 🔐 STEP 9 COMPLIANT DUAL RECOVERY FLOW (SECURITY QUESTION & OTP RELAY)
        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")

            if st.session_state.forgot_stage == "init":
                email_target = st.text_input("Registered email address", placeholder="you@infosys.com").lower().strip()
                st.markdown("<br>", unsafe_allow_html=True)

                col_sq, col_otp = st.columns(2)
                if col_sq.button("Via Security Question", use_container_width=True, key="forgot_route_sq_btn"):
                    if not email_target: st.error("⚠️ Please enter your email address.")
                    else:
                        with get_db() as c: r = c.execute("SELECT security_question FROM users WHERE email=?", (email_target,)).fetchone()
                        if r:
                            st.session_state.reset_email = email_target
                            st.session_state.sq_p = r[0]
                            st.session_state.reset_mode = "sq"
                            st.session_state.forgot_stage = "challenge"
                            st.rerun()
                        else: st.error("❌ Registered email address not found.")

                if col_otp.button("Via OTP Mail Relay", use_container_width=True, key="forgot_route_otp_btn"):
                    if not email_target: st.error("⚠️ Please enter your email address.")
                    elif not EMAIL_PASSWORD:
                        st.error("❌ Gmail App Password missing! Put `EMAIL_PASSWORD` inside Colab Secrets.")
                    else:
                        with get_db() as c: r = c.execute("SELECT 1 FROM users WHERE email=?", (email_target,)).fetchone()
                        if r:
                            otp_code = generate_otp()
                            with st.spinner("Dispatching verification OTP over secure SMTP relays..."):
                                ok, msg = send_professional_email(email_target, otp_code, EMAIL_PASSWORD)
                            if ok:
                                st.session_state.reset_email = email_target
                                st.session_state.jwt_otp_token = make_otp_token(email_target, otp_code)
                                st.session_state.reset_mode = "otp"
                                st.session_state.forgot_stage = "challenge"
                                st.success("✅ Code dispatched! Check your inbox.")
                                time.sleep(1)
                                st.rerun()
                            else: st.error(f"❌ {msg}")
                        else: st.error("❌ Registered email address not found.")

            else:
                # --- PROCESS ROUTE A (SECURITY CHALLENGE) ---
                if st.session_state.reset_mode == "sq":
                    st.info(f"❓ **Security Question Challenge:** {st.session_state.sq_p}")
                    ans = st.text_input("Your answer", placeholder="Enter answer given at registration").lower().strip()
                    npw = st.text_input("New password", type="password", placeholder="Min. 8 characters")
                    confirm_npw = st.text_input("Confirm new password", type="password", placeholder="Re-enter password")
                    st.markdown("<br>", unsafe_allow_html=True)

                    if st.button("Reset Password via Question →", use_container_width=True, key="sq_reset_execute_btn"):
                        if not ans or not npw or not confirm_npw: st.error("⚠️ All fields are mandatory.")
                        elif not validate_password(npw): st.error("❌ Password failure. Must fulfill Step 8 criteria rules.")
                        elif npw != confirm_npw: st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c: r = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                            if r and check_txt(ans, r[0]):
                                with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                st.success("✅ Credentials verified. Password updated successfully!")
                                time.sleep(1.5)
                                st.session_state.reset_email = None
                                st.session_state.forgot_stage = "init"
                                navigate("Login")
                            else: st.error("❌ Incorrect security answer.")

                # --- PROCESS ROUTE B (OTP TOKEN CHALLENGE) ---
                elif st.session_state.reset_mode == "otp":
                    st.info(f"📧 Code sent to **{st.session_state.reset_email}** (Expires in {OTP_EXPIRY_MINUTES} mins).")
                    otp_input = st.text_input("6-Digit OTP Code", placeholder="e.g. 123456").strip()
                    npw = st.text_input("New password", type="password", placeholder="Min. 8 characters")
                    confirm_npw = st.text_input("Confirm new password", type="password", placeholder="Re-enter password")
                    st.markdown("<br>", unsafe_allow_html=True)

                    if st.button("Verify OTP & Reset Password →", use_container_width=True, key="otp_reset_execute_btn"):
                        if not otp_input or not npw or not confirm_npw: st.error("⚠️ All fields are mandatory.")
                        elif not validate_password(npw): st.error("❌ Password failure. Must fulfill Step 8 criteria rules.")
                        elif npw != confirm_npw: st.error("❌ Passwords do not match.")
                        else:
                            ok, msg = verify_otp_token(st.session_state.jwt_otp_token, otp_input, st.session_state.reset_email)
                            if ok:
                                with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                st.success("🎉 OTP verified. Account credentials updated successfully!")
                                time.sleep(1.5)
                                st.session_state.reset_email = None
                                st.session_state.forgot_stage = "init"
                                navigate("Login")
                            else: st.error(f"❌ {msg}")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel Recovery Process", use_container_width=True, key="cancel_recovery_btn"):
                st.session_state.reset_email = None
                st.session_state.reset_mode = None
                st.session_state.forgot_stage = "init"
                navigate("Login")

# ============================================================
# 📊 8. AUTHENTICATED PORTALS (STEP 10 & 11 SEPARATION)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    current_email = payload["email"]
    with get_db() as c: user_row = c.execute("SELECT username FROM users WHERE email=?", (current_email,)).fetchone()
    uname = user_row[0] if user_row else "Core User"

    # 🚀 SIDEBAR NAVIGATION WORKSPACE & LOGOUT GATE
    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⚡</div>
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Infosys Portal</div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Management Panel" if current_email=="infosys@ai" else "Intelligent Analytics"}</div>
        </div><hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Settings", "Logout"] if current_email=="infosys@ai" else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "gear", "box-arrow-right"] if current_email=="infosys@ai" else ["house", "graph-up", "file-text", "box-arrow-right"],
                           styles={"container": {"background-color": COLORS['bg_sidebar']}, "nav-link-selected": {"background-color": COLORS['accent'], "color": COLORS['accent_text']}})

        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.success("🔒 Session cleared safely. Returning to sign in screen...")
            time.sleep(1)
            st.rerun()

    # 🛡️ STEP 11: PRIVILEGED ADMINISTRATIVE MASTER DASHBOARD
    if current_email == "infosys@ai":
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        st.subheader("👥 Registered Platform Accounts Ledger")

        # Read-only fetch: Username/Email only. Never select password_hash!
        with get_db() as c: user_data = c.execute("SELECT id, username, email FROM users").fetchall()

        if user_data:
            st.markdown(f"""
            <div style="background:#ffffff; border:2px solid {COLORS['border_light']}; border-radius:12px; padding:24px;">
                <div style="display:grid; grid-template-columns: 1fr 3fr 4fr; font-weight:700; color:{COLORS['text_heading']}; border-bottom:2px solid #e2e8f0; padding-bottom:10px; margin-bottom:12px;">
                    <div>User ID</div>
                    <div>Username</div>
                    <div>Email Endpoint Address</div>
                </div>
            """, unsafe_allow_html=True)

            for row in user_data:
                col_id, col_un, col_em = st.columns([1, 3, 4])
                col_id.markdown(f"**`#{row[0]}`**")
                col_un.write(row[1])
                col_em.write(row[2])
                st.markdown("<hr style='margin:8px 0; border-color:#f1f5f9;'>", unsafe_allow_html=True)

            st.markdown("</div>", unsafe_allow_html=True)
            st.markdown("<br>", unsafe_allow_html=True)
            st.caption(f"📊 Total System Accounts Registered: **{len(user_data)}**")
        else:
            st.info("ℹ️ No users registered inside the database yet.")

    # 📈 STEP 10: METRIC ANALYTICS STANDARD PORTAL VIEWPORT
    else:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Analytics Dashboard</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:28px;">{icon}</div>
                <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['accent']}, "bgcolor": COLORS['bg_card_alt'], "borderwidth": 1, "bordercolor": COLORS['border']}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)

Overwriting app.py


In [29]:
import os, time, subprocess, sys
from pyngrok import ngrok
from google.colab import userdata

# 1. Safely retrieve secure keys from Colab Secrets
try:
    os.environ['EMAIL_PASSWORD'] = userdata.get('EMAIL_PASSWORD')
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
    if ngrok_token: ngrok.set_auth_token(ngrok_token)
except Exception as e:
    print(f"⚠️ Secret Error: Make sure EMAIL_PASSWORD and NGROK_AUTHTOKEN exist in Colab Secrets.")

# 2. Shut down stuck Streamlit instances to refresh the networking stack
!pkill -f streamlit
time.sleep(2)

# 3. Start your newly merged app cleanly in the background
# Removed stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL to show Streamlit output for debugging
process = subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'app.py', '--server.port', '8501'],
                         env=os.environ.copy())
time.sleep(20) # Increased sleep to ensure Streamlit is ready

# 4. Safely connect or reuse your public Ngrok domain endpoint
try:
    tunnels = ngrok.get_tunnels()
    if tunnels:
        public_url = tunnels[0].public_url
        print("♻️ Reusing existing active Ngrok tunnel...")
    else:
        public_url = ngrok.connect(8501).public_url
        print("🚀 Created new Ngrok tunnel...")

    print("=" * 70)
    print(f"👉 COMBINED PORTAL LIVE URL: {public_url}")
    print("=" * 70)
    print("⏳ Server running! Press the Colab Stop button or [Ctrl + C] to shut down.")
    while True: time.sleep(1)

except Exception as e:
    print(f"⚠️ Notice: {e}")
    print("🔄 Waiting 5 seconds for Ngrok cloud servers to release domain lock...")
    ngrok.kill()
    !pkill -f ngrok
    time.sleep(5)
    public_url = ngrok.connect(8501).public_url
    print("=" * 70)
    print(f"👉 COMBINED PORTAL LIVE URL: {public_url}")
    print("=" * 70)
    while True: time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Shutting down server...")
    !pkill -f streamlit

🚀 Created new Ngrok tunnel...
👉 COMBINED PORTAL LIVE URL: https://overdrawn-pebbly-defuse.ngrok-free.dev
⏳ Server running! Press the Colab Stop button or [Ctrl + C] to shut down.

🛑 Shutting down server...
